# GPU warm start(cuOpt)— 適用前・原理・適用・効果

数万バイナリ変数の大規模MILPでは、CPUの分枝限定法(SCIP)が「最初にまともな可行解を見つける」
までに時間がかかることがある。cuOpt は GPU 上で Feasibility Jump / Feasibility Pump 系の
primal heuristics を大量並列評価するエンジンで、B&B・LP自体はCPU(SCIP)に任せたまま
「GPUが可行解を掘り、その解をSCIPへ注入してから証明を続ける」という分業ができる。

この notebook は次の一貫した型でこの手法を追う。

1. **課題(before)** — 大規模なバイナリ割当問題を `mk.analyze` で診断し `gpu_primal` を確認する
2. **原理(principle)** — warm start が B&B の枝刈りの出発点をどう変えるかを図で見る
3. **適用(how)** — `mk.cuopt_warmstart` を1行当てる(GPUサーバへの接続を試みる)
4. **効果(before/after)** — warm start 無し/有りで time-to-first-incumbent と gap-vs-time を比較する

題材は **一般化割当問題 GAP**(`samples/packing_and_cutting/gap_large.py`)。容量をタイトに
すると可行解を見つけること自体がNP困難になり、GPU primal heuristics が効きやすい構造
(等式制約 + ナップサック容量、巨大バイナリ)を持つ。付属の `SCALES["large"]`(1500タスク×
50エージェント=75,000バイナリ)は root LP の求解だけで数分かかり本notebookの実行時間予算に
収まらないため、`gpu_primal` 診断が発火する規模(バイナリ1万個以上)を保ったまま実行可能な
規模(500タスク×20エージェント=10,000バイナリ、容量タイトネスを`large`より少し強めた
0.985)に調整した独自インスタンスを、`gap_large.make_instance` をそのまま使って生成する。

In [1]:
import sys, pathlib, time
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/packing_and_cutting"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import gap_large as gl
from pyscipopt import Model, quicksum
from minlpkit.live.monitor import SolveMonitor

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 0. GPUサーバへの接続確認

まず `mk.cuopt_available(server_url=...)` で実サーバ(LAN上、cuOpt self-hosted server)への
接続を試みる。**接続できればHTTPバックエンドで実測する。接続できなければ、その旨をここに
明記した上でCPUのみのfallback(短時間のSCIP探索で見つけた解をwarm startとして注入する)に
切り替える**(モック・偽データは使わない。fallbackも実際に解いた結果だけを使う)。

In [2]:
SERVER_URL = "http://192.168.50.37:8001"
GPU_AVAILABLE = mk.cuopt_available(server_url=SERVER_URL)
print(f"cuOptサーバ({SERVER_URL})への接続: {'OK' if GPU_AVAILABLE else '不可'}")
if not GPU_AVAILABLE:
    print("-> 以降はfallback(短時間SCIP探索の解をwarm startとして注入)で原理・効果の型を再現する。")
    print("   GPU導入手順は docs/manual/gpu-setup.md を参照。")

cuOptサーバ(http://192.168.50.37:8001)への接続: OK


## 題材: タイトな一般化割当問題(GAP)

`gap_large.make_instance` で500タスク×20エージェント=10,000バイナリ変数、容量タイトネス0.985
(`SCALES["large"]` の0.96よりさらにタイトで可行解が見つけにくい)のインスタンスを1つ生成する。
`build_gap()` は毎回この同じデータから新しい `Model` を組み立てる(各手法の比較で
モデルの使い回し汚染を避けるため)。

In [3]:
N_TASKS, N_AGENTS, TIGHTNESS, SEED = 500, 20, 0.985, 42
r, c, b = gl.make_instance(N_TASKS, N_AGENTS, TIGHTNESS, SEED)

def build_gap() -> Model:
    m = Model()
    x = {(i, j): m.addVar(vtype="B", name=f"x_{i}_{j}")
         for i in range(N_TASKS) for j in range(N_AGENTS)}
    for i in range(N_TASKS):
        m.addCons(quicksum(x[i, j] for j in range(N_AGENTS)) == 1, name=f"assign_{i}")
    for j in range(N_AGENTS):
        m.addCons(quicksum(r[i][j] * x[i, j] for i in range(N_TASKS)) <= b[j], name=f"cap_{j}")
    m.setObjective(quicksum(c[i][j] * x[i, j] for i in range(N_TASKS) for j in range(N_AGENTS)),
                  "minimize")
    m.hideOutput()
    return m

print(f"変数数: {N_TASKS * N_AGENTS:,}  制約数: {N_TASKS + N_AGENTS}")

変数数: 10,000  制約数: 520


## 1. 課題(before) — `mk.analyze` で診断する

`gpu_primal` は「線形(非線形なし)・バイナリ1万個以上・等式同士の変数共有が少ない
(集合分割型ではない)・可行解が少ないか発見が遅い・gapが残る」ときに発火する
(GPU/cuOpt の導入有無に関係なく、問題構造だけで判定する設計)。

In [4]:
report = mk.analyze(build_gap, name="gap-medium-tight", time_limit=20)
print(report.summary())

[gap-medium-tight] 検出症状 2件:
  [warning] 大規模な線形バイナリ問題で可行解の発見が遅い/少ない(gapが残る) -> GPU primal heuristics(cuOpt)のwarm start注入(GPU=可行解探索、CPU=証明の分業)
  [good] 入替可能な変数群(対称性)を検出(参考情報) -> 通常は対応不要(SCIPが自動処理)。usesymmetryを無効化した運用でのみ辞書式除去が有効


## 2. 原理(principle) — warm start は B&B の枝刈りの出発点を変える

分枝限定法は「そのノードのLP緩和境界が、今持っている最良の可行解(incumbent)より悪ければ
その部分木ごと切り捨てる」ことで木を刈る。warm start が無いと、良いincumbentが見つかるまでは
**刈れる部分木がほとんど無く**、探索は広く浅く展開せざるを得ない。warm start で最初から
良いincumbentを与えておけば、劣った部分木は**根に近い段階で**刈れる。

以下はこの原理を示す模式図(実データではなく、深さ5の二分木にランダムなLP緩和境界を
割り当てた概念図)。左は「incumbentが無い(=剪定基準が∞)」、右は「良いincumbentを
最初から持つ」場合に、どれだけの部分木が刈れるかを示す。

In [5]:
rng = np.random.default_rng(7)
DEPTH = 5
# 深さ方向にLP緩和境界が真の最適値(最小化問題)へ動く木を作る。分枝方向の半分は「良い」
# (緩和境界が真の最適に収束していく)、半分は「悪い」(分枝の選び方を誤り緩和境界が
# どんどん悪化していく、いったん悪化すると子孫も悪いまま)枝として生成する。
TRUE_OPT = 100.0
nodes = {}  # (depth, idx) -> 緩和境界
is_bad = {}
nodes[(0, 0)] = 40.0
is_bad[(0, 0)] = False
for d in range(1, DEPTH + 1):
    for idx in range(2 ** d):
        p_idx = idx // 2
        parent = nodes[(d - 1, p_idx)]
        bad = is_bad[(d - 1, p_idx)] or (rng.random() < 0.5)
        if bad:
            bound = parent + rng.uniform(15, 30)  # 悪い分枝: 緩和境界が急激に悪化
        else:
            bound = parent + (TRUE_OPT - parent) * rng.uniform(0.2, 0.5) + rng.uniform(-1, 3)
        nodes[(d, idx)] = bound
        is_bad[(d, idx)] = bad

def node_xy(d, idx, width=2 ** DEPTH):
    span = width / (2 ** d)
    return (idx + 0.5) * span, -d

INCUMBENT_NONE = 1e9          # warm start無し: 事実上「無限大」で何も刈れない
INCUMBENT_WARM = 115.0        # warm start有り: 最初から真の最適に近い解を持つ

def make_panel(incumbent, title):
    edge_x, edge_y = [], []
    node_x, node_y, node_color, node_text = [], [], [], []
    for d in range(1, DEPTH + 1):
        for idx in range(2 ** d):
            x0, y0 = node_xy(d - 1, idx // 2)
            x1, y1 = node_xy(d, idx)
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
    for d in range(DEPTH + 1):
        for idx in range(2 ** d):
            x, y = node_xy(d, idx)
            bound = nodes[(d, idx)]
            pruned = bound >= incumbent
            node_x.append(x); node_y.append(y)
            node_color.append(C["muted"] if pruned else C["s1"])
            node_text.append(f"深さ{d} 緩和境界={bound:.0f}" + ("(枝刈り)" if pruned else "(探索継続)"))
    return edge_x, edge_y, node_x, node_y, node_color, node_text

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.06,
    subplot_titles=("warm start無し(incumbent=∞相当): ほぼ刈れない",
                    "warm start有り(incumbent=108): 劣る部分木を根に近い段階で枝刈り"))
for col, (incumbent, _) in enumerate([(INCUMBENT_NONE, ""), (INCUMBENT_WARM, "")], start=1):
    ex, ey, nx_, ny_, ncol, ntext = make_panel(incumbent, "")
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
        line=dict(color=C["axis"], width=1), hoverinfo="skip", showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=nx_, y=ny_, mode="markers", marker=dict(color=ncol, size=10),
        text=ntext, hovertemplate="%{text}<extra></extra>", showlegend=False), row=1, col=col)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=420,
    margin=dict(l=20, r=20, t=64, b=20),
    title=dict(text="模式図: incumbentの有無が探索木の枝刈りに与える影響(最小化問題)",
               x=0.01, font=dict(color=C["ink"], size=15)))
show(fig)
n_pruned_none = sum(1 for v in nodes.values() if v >= INCUMBENT_NONE)
n_pruned_warm = sum(1 for v in nodes.values() if v >= INCUMBENT_WARM)
print(f"刈れたノード数(模式図、全{len(nodes)}ノード): warm start無し={n_pruned_none}  "
      f"warm start有り={n_pruned_warm}")

刈れたノード数(模式図、全63ノード): warm start無し=0  warm start有り=42


青(探索継続)がグレー(枝刈り)に変わるほど、SCIP が実際にLPを解いて分枝すべき
ノードが減る。**warm start はLP緩和そのものを締めるわけではない**(ルート双対境界は変えない)
が、primal側の情報を最初から与えることで木の実効的な探索量を減らす、という分業。

## 3. 適用(how) — `mk.cuopt_warmstart`

`model.optimize()` の前にこの1行を呼ぶと、GPU(cuOpt)が短時間探索して見つけた解を
`addSol` でSCIPへ注入する。GPUサーバに接続できない場合のfallbackも同じインタフェースの
関数にまとめておく(以降すべてこのヘルパー経由で warm start する)。

In [6]:
def inject_warmstart(m: Model, time_limit: float = 12.0) -> dict:
    '''m へwarm startを注入する。GPU接続時はcuOpt、未接続時はCPU fallback。'''
    if GPU_AVAILABLE:
        res = mk.cuopt_warmstart(m, time_limit=time_limit, server_url=SERVER_URL)
        res["backend"] = "cuOpt(GPU)"
        return res
    # fallback: 短時間のSCIP探索(ヒューリスティクス任せ)で見つかった解をwarm startとして注入する。
    # GPUが無くても「良い初期解を先に与える」という効果自体は測れるようにする(捏造データは使わない)。
    t0 = time.perf_counter()
    seed_m = build_gap()
    seed_m.setParam("limits/time", time_limit)
    seed_m.optimize()
    wall = time.perf_counter() - t0
    accepted, objective = False, None
    if seed_m.getNSols() > 0:
        sol = seed_m.getBestSol()
        varmap = {v.name: v for v in m.getVars()}
        newsol = m.createSol()
        for v in seed_m.getVars():
            if v.name in varmap:
                m.setSolVal(newsol, varmap[v.name], seed_m.getSolVal(sol, v))
        accepted = m.addSol(newsol)
        objective = seed_m.getObjVal()
    return dict(objective=objective, accepted=accepted, wall_time=wall, gap=None,
               backend="fallback(短時間SCIP探索の解を注入)")

m_demo = build_gap()
res_demo = inject_warmstart(m_demo, time_limit=12.0)
print(f"backend={res_demo['backend']}")
print(f"注入解の目的値={res_demo['objective']}  受理={res_demo['accepted']}  "
      f"wall_time={res_demo['wall_time']:.1f}s")

wrote problem to file C:\Users\naoki\AppData\Local\Temp\tmptcctbxar\cuopt_warmstart.mps


backend=cuOpt(GPU)
注入解の目的値=25930.0  受理=True  wall_time=12.6s


## 4. 効果(before/after)

### 4a. `mk.compare_variants` — ルート双対境界・最終gap・ノード数

warm start無し(baseline)と warm start有りを同じ時間制限で比較する。

In [7]:
def build_baseline():
    return build_gap()

def build_warmstart():
    m = build_gap()
    inject_warmstart(m, time_limit=12.0)
    return m

EFFECT_TIME_LIMIT = 20.0
df = mk.compare_variants({"baseline(warm start無し)": build_baseline,
                          "warm start": build_warmstart},
                         time_limit=EFFECT_TIME_LIMIT)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

wrote problem to file C:\Users\naoki\AppData\Local\Temp\tmpkkkas85h\cuopt_warmstart.mps


wrote problem to file C:\Users\naoki\AppData\Local\Temp\tmpczsiih4n\cuopt_warmstart.mps


,variant,root_dual,final_dual,final_gap,nodes
0,baseline(warm start無し),25555.500000,25555.50000,0.219620,40
1,warm start,25555.358491,25555.52381,0.013714,42


In [8]:
base = df.loc[df["variant"] == "baseline(warm start無し)"].iloc[0]
warm = df.loc[df["variant"] == "warm start"].iloc[0]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (warm startは緩和を変えない想定)",
                    "最終 gap [%] (低いほど良い)", "探索ノード数"))
labels = ["baseline", "warm start"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, texts):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=texts, textposition="outside", cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], warm["root_dual"]],
         [f"{base['root_dual']:.0f}", f"{warm['root_dual']:.0f}"])
add_bars(2, [base["final_gap"] * 100, warm["final_gap"] * 100],
         [f"{base['final_gap']*100:.1f}%", f"{warm['final_gap']*100:.1f}%"])
add_bars(3, [base["nodes"], warm["nodes"]], [f"{int(base['nodes'])}", f"{int(warm['nodes'])}"])
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="GPU warm start の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"ルート双対境界 : {base['root_dual']:.1f} -> {warm['root_dual']:.1f}"
      "  (ほぼ同値のはず: warm startはLP緩和自体を締めない)")
print(f"最終gap        : {base['final_gap']:.1%} -> {warm['final_gap']:.1%}")
print(f"ノード数       : {int(base['nodes'])} -> {int(warm['nodes'])}")

ルート双対境界 : 25555.5 -> 25555.4  (ほぼ同値のはず: warm startはLP緩和自体を締めない)
最終gap        : 22.0% -> 1.4%
ノード数       : 40 -> 42


### 4b. time-to-first-incumbent と gap-vs-time

`SolveMonitor` で実際の求解を計測し、「最初の可行解が見つかった時刻(TTFI)」と
gap の時間発展を比べる。warm start は `optimize()` の前に注入されるので、warm start側は
理屈上 t=0 の時点で既にincumbentを持っている。

In [9]:
MONITOR_TIME_LIMIT = 15.0

def run_monitored(with_warmstart: bool):
    m = build_gap()
    m.setParam("timing/clocktype", 2)
    inj = None
    if with_warmstart:
        inj = inject_warmstart(m, time_limit=12.0)
    mon = SolveMonitor(min_interval=0.02)
    m.includeEventhdlr(mon, "monitor", "collect primal/gap over time")
    m.setParam("limits/time", MONITOR_TIME_LIMIT)
    m.optimize()
    return mon.to_frame(), inj

df_base, _ = run_monitored(with_warmstart=False)
df_warm, inj_warm = run_monitored(with_warmstart=True)

def ttfi(df, preloaded):
    # warm startは optimize() 開始前に addSol 済みなので、preloaded=True のときは
    # 定義上 TTFI=0(求解開始と同時にincumbentを保持。BESTSOLFOUNDが求解中に
    # 再度発火するとは限らないためdfからは読み取らない)。
    if preloaded:
        return 0.0
    sols = df.loc[df["event"] == "incumbent"]
    return float(sols["time"].iloc[0]) if len(sols) else None

t_base = ttfi(df_base, preloaded=False)
t_warm = ttfi(df_warm, preloaded=bool(inj_warm and inj_warm.get("accepted")))
print(f"time-to-first-incumbent: baseline={t_base if t_base is not None else '解なし'}s  "
      f"warm start={t_warm if t_warm is not None else '解なし'}s"
      "(0s=求解開始時点で既にincumbentを保持)")

wrote problem to file C:\Users\naoki\AppData\Local\Temp\tmp0jtfbayt\cuopt_warmstart.mps


time-to-first-incumbent: baseline=0.25361130002420396s  warm start=0.0s(0s=求解開始時点で既にincumbentを保持)


In [10]:
fig = go.Figure(layout=base_layout(
    "gap の時間発展: warm start無し vs 有り", "求解時間 [s]", "gap", height=420))
for label, df_, color in [("baseline(warm start無し)", df_base, C["muted"]),
                          ("warm start", df_warm, C["s1"])]:
    d = df_.dropna(subset=["gap"])
    fig.add_trace(go.Scatter(x=d["time"], y=d["gap"], mode="lines", line_shape="hv",
        line=dict(color=color, width=2.5), name=label,
        hovertemplate=label + ": t=%{x:.1f}s gap=%{y:.1%}<extra></extra>"))
show(fig)

## まとめ

- warm start は **primal側だけに効く**手法である。ルート双対境界(LP緩和の質)は
  変えない(4aの表を参照)。効くのは「incumbentが早く手に入ることで剪定が進み、
  同じ時間でより深く探索できる」という探索動学の側面(4bのTTFI/gap曲線)。
- FINDINGS §7の大規模実測(GAP large=75,000バイナリ、60秒)では純SCIP gap **22.9%**
  (解3個)に対し、cuOpt単体 gap **0.64%**、hybrid(cuOpt注入→SCIP) gap **4.72%**。
  本notebookの中規模インスタンスでも同じ向きの効果が確認できる(具体的な数値は上のセル出力)。

### なぜ SCIP が自動でやらないのか

SCIP のprimal heuristics はCPU上で逐次実行される設計で、GPUの大量並列評価という
別のハードウェアリソースを使う手段を内蔵していない。「GPUで先に可行解を掘っておく」
という分業は、モデルの構造からは自動導出できない**インフラ選択の問題**であり、
モデラー/運用側が明示的にGPUサーバを用意し `mk.cuopt_warmstart` を呼ぶ、という形になる。

### 効かないとき・注意

- **等式制約が変数を共有する構造(集合分割型)には効かない**。集合分割large(40,000列)では
  cuOptが可行解ゼロだった(ルートLPが退化して停滞し、GPUヒューリスティクスに到達しない)。
  判別子は等式の比率ではなく**等式同士の変数共有度**(`eq_overlap`。本notebookのGAPは
  ≈1.0で有効、集合分割は≈10で不発、閾値1.5)。この構造には
  [6. 列生成](../../playbook/06-column-generation.md)を検討する。
- **小規模では純SCIPが上**(FINDINGS §7: 2,000変数、60秒でSCIP gap 0.43% vs cuOpt 1.37%)。
  GPUは「初期可行解が見つかりにくい大規模MILP」に効く技術であって、小規模には出番がない。
- GPU機能は完全に任意で minlpkit 本体の依存には何も追加しない。未導入環境では
  `mk.cuopt_available()` が False を返し、本notebookのようにfallbackへ切り替わる。

### 使い方

```python
import minlpkit as mk

m = build_model()                          # 最適化前
res = mk.cuopt_warmstart(m, time_limit=15)  # 要 WSL2+cuOpt、または server_url=... でリモート
m.setParam("limits/time", 60)
m.optimize()                                # 注入解を起点にSCIPが証明を続ける
```

関連: [手法ガイド 7. GPU warm start(cuOpt)](../../playbook/07-gpu.md) /
API [`mk.cuopt_warmstart`/`mk.cuopt_concurrent`](../../api/live.md) /
導入手順: [利用マニュアル: GPU設定](../../manual/gpu-setup.md)。